# Scaled dot-product attention

Implement attention with NumPy, visualize its weights, and test that a causal mask blocks information from future tokens. This explains the mechanism; it is not a trained language model.

**Prerequisites:** matrix multiplication and softmax. **Environment:** base requirements. **Execution:** restart and run all cells. No data download, framework, or GPU is required.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
Q = rng.normal(size=(5, 4))
K = rng.normal(size=(5, 4))
V = rng.normal(size=(5, 3))
print('Q, K, V shapes:', Q.shape, K.shape, V.shape)


## Scores, stable softmax, and weighted values

Attention is $\operatorname{softmax}(QK^T/\sqrt{d_k})V$. Each query receives weights over key positions. Subtracting the row maximum before exponentiation leaves softmax unchanged and reduces overflow risk. This function handles a single matrix of queries/keys, without a batch or head dimension.


In [ ]:
def attention(query, key, value, allowed=None):
    query, key, value = [np.asarray(array, dtype=float) for array in (query, key, value)]
    if any(array.ndim != 2 for array in (query, key, value)):
        raise ValueError('Expected rank-2 query, key, and value matrices')
    if query.shape[1] != key.shape[1] or key.shape[0] != value.shape[0] or key.shape[1] == 0:
        raise ValueError('Incompatible attention dimensions')
    if not all(np.isfinite(array).all() for array in (query, key, value)):
        raise ValueError('Inputs must be finite')
    scores = query @ key.T / np.sqrt(key.shape[1])
    if allowed is not None:
        allowed = np.asarray(allowed, dtype=bool)
        if allowed.shape != scores.shape or not allowed.any(axis=1).all():
            raise ValueError('Mask shape must match scores; every query needs an allowed key')
        scores = np.where(allowed, scores, -np.inf)
    exponentials = np.exp(scores - scores.max(axis=1, keepdims=True))
    weights = exponentials / exponentials.sum(axis=1, keepdims=True)
    return weights @ value, weights

output, weights = attention(Q, K, V)
assert output.shape == (5, 3)
np.testing.assert_allclose(weights.sum(axis=1), 1)
print('First query weights:', np.round(weights[0], 3))


## A causal mask allows only the current and earlier positions

The lower triangular mask ensures position $i$ cannot attend to keys at $j>i$. Fully masked rows are rejected because their softmax would be undefined. An encoder for classification often uses unrestricted attention; a causal language-model decoder uses a causal mask.


In [ ]:
causal = np.tril(np.ones((len(Q), len(K)), dtype=bool))
causal_output, causal_weights = attention(Q, K, V, allowed=causal)
np.testing.assert_allclose(causal_weights.sum(axis=1), 1)
assert np.all(causal_weights[np.triu_indices(5, k=1)] == 0)

# A future key/value perturbation must not change outputs at positions 0, 1, or 2.
changed_key, changed_value = K.copy(), V.copy()
changed_key[3:] += 100
changed_value[3:] -= 100
changed_output, _ = attention(Q, changed_key, changed_value, allowed=causal)
np.testing.assert_allclose(changed_output[:3], causal_output[:3], rtol=0, atol=0)
print('Future-token perturbation leaves past outputs unchanged')

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for ax, matrix, title in zip(axes, [weights, causal_weights], ['Unrestricted', 'Causal']):
    plotted = ax.imshow(matrix, vmin=0, vmax=1, cmap='viridis')
    ax.set(title=title, xlabel='Key position', ylabel='Query position')
fig.colorbar(plotted, ax=axes.ravel().tolist(), label='Weight')
plt.show()


## Practice

Set all queries and keys to zero and predict the weights before running the cell: unrestricted attention should average all values. Then try a causal mask and explain why the number of averaged values changes by row. Attention alone does not encode sequence position; the next lesson adds positional information in a Transformer encoder.

**References:** [Attention Is All You Need](https://arxiv.org/abs/1706.03762), [PyTorch scaled dot-product attention](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html).
